# 第 43 课｜yfinance 数据适配与复权口径

本 Notebook 对应「365 天量化金融」第 43 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为默认离线、可核对、可重复执行的量化教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/43_youtube-c-HzVf8XECk/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟行情；接口结构仿照 yfinance，非 Yahoo 真实数据
- 状态：可运行学习稿；不代表策略收益承诺

> 网络接口和研究计算分离：Notebook 默认不请求第三方服务。真实接口只作为适配层示意，计算使用带来源声明的本地快照，避免限流、字段变化和历史数据重算破坏复现。

## 本节量化问题

`yfinance.download` 的难点不是函数调用，而是三件事：批量下载的 MultiIndex 如何按标签读取、拆股/分红后应该使用哪种价格、接口失败如何形成可追溯状态。

本节输出：

1. 一个 yfinance 风格的 `字段 × 股票` MultiIndex 快照；
2. 原始收盘价与复权收盘价的拆股对照；
3. 带版本、参数、哈希和逐标的状态的缓存清单。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 1. 批量结果的 MultiIndex 不是普通二维列

yfinance 批量下载常把列组织为两层：第一层是 `Close/Volume` 等字段，第二层是股票代码。应使用 `vendor["Close"]` 按字段标签取得“日期 × 股票”矩阵，不能假设某一列位置永远是收盘价。

In [2]:
# 把本地长表重排为 yfinance 风格的两层列，并显示原始结构。
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
sample = daily.loc[daily["symbol"].isin(["BANK_A", "EV_A", "TECH_A"])].copy()
parts = {
    "Close": sample.pivot(index="date", columns="symbol", values="close"),
    "Adj Close": sample.pivot(index="date", columns="symbol", values="adj_close"),
    "Volume": sample.pivot(index="date", columns="symbol", values="volume"),
}
vendor = pd.concat(parts, axis=1).sort_index()
vendor.columns.names = ["Price", "Ticker"]

print("列层级:", vendor.columns.names)
display(vendor.head(3))

close = vendor["Close"]
assert close.shape == (756, 3)
assert close.columns.tolist() == ["BANK_A", "EV_A", "TECH_A"]

列层级: ['Price', 'Ticker']


Price         Close                   Adj Close                     Volume  \
Ticker       BANK_A     EV_A   TECH_A    BANK_A     EV_A   TECH_A   BANK_A   
date                                                                         
2023-01-03  11.7218  27.5575  36.2134   11.7218  27.5575  36.2134  2822797   
2023-01-04  11.6934  27.5189  36.5943   11.6934  27.5189  36.5943  1999548   
2023-01-05  11.6864  27.8166  36.0263   11.6864  27.8166  36.0263  1974821   

Price                         
Ticker         EV_A   TECH_A  
date                          
2023-01-03  5113762  6241108  
2023-01-04  4037866  3498641  
2023-01-05  3392470  2533583

## 2. 复权价格处理拆股造成的机械跳变

下面在虚构资产上构造 2 拆 1：拆股日之后每股原始价格减半，但持股数翻倍，经济价值没有因此腰斩。`Adjusted Close` 把历史序列调整到可连续比较的口径。

标准化净值为 `NAV_t = P_t / P_0`。只有价格口径一致时，标准化曲线才可比较。

In [3]:
# 构造拆股事件，比较原始价格收益与复权价格收益。
base = close["TECH_A"].iloc[:80].copy()
split_date = base.index[40]
raw_close = base.copy()
raw_close.loc[split_date:] = raw_close.loc[split_date:] / 2
adjusted_close = base.copy()
actions = pd.DataFrame({"Stock Splits": [2.0], "Dividends": [0.0]}, index=[split_date])

comparison = pd.DataFrame({
    "raw_close": raw_close,
    "adjusted_close": adjusted_close,
})
comparison["raw_return"] = comparison["raw_close"].pct_change()
comparison["adjusted_return"] = comparison["adjusted_close"].pct_change()
comparison["adjusted_nav"] = comparison["adjusted_close"] / comparison["adjusted_close"].iloc[0]

display(actions)
display(comparison.loc[split_date - pd.Timedelta(days=3): split_date + pd.Timedelta(days=3)])
assert comparison.loc[split_date, "raw_return"] < -0.45
assert abs(comparison.loc[split_date, "adjusted_return"]) < 0.10

,Stock Splits,Dividends
2023-02-28,2.0,0.0


,raw_close,adjusted_close,raw_return,adjusted_return,adjusted_nav
date,,,,,
2023-02-27,34.11030,34.1103,-0.012764,-0.012764,0.941925
2023-02-28,17.35940,34.7188,-0.491080,0.017839,0.958728
2023-03-01,17.59060,35.1812,0.013318,0.013318,0.971497
2023-03-02,17.38065,34.7613,-0.011935,-0.011935,0.959902
2023-03-03,17.77870,35.5574,0.022902,0.022902,0.981885


## 3. 缓存必须同时保存请求参数与失败状态

同一文件名不能说明使用了哪个 yfinance 版本、是否 `auto_adjust=True`、哪些代码为空。这里把参数、数据范围和 SHA-256 一起写入清单。真实适配器应对限流和空表分别记录，而不是把失败资产悄悄删掉。

In [4]:
# 生成可追溯的离线缓存和逐标的状态。
import hashlib
import importlib.metadata as metadata

cache_path = OUTPUT_DIR / "43_yfinance_style_snapshot.csv"
vendor.to_csv(cache_path)
digest = hashlib.sha256(cache_path.read_bytes()).hexdigest()
status = pd.DataFrame([
    {"ticker": ticker, "status": "ok", "rows": int(close[ticker].notna().sum())}
    for ticker in close.columns
] + [{"ticker": "MISSING_DEMO", "status": "empty", "rows": 0}])
manifest = {
    "adapter": "yfinance-style-offline",
    "yfinance_version": metadata.version("yfinance"),
    "auto_adjust": False,
    "price_fields": ["Close", "Adj Close", "Volume"],
    "date_start": str(vendor.index.min().date()),
    "date_end": str(vendor.index.max().date()),
    "sha256": digest,
}
(OUTPUT_DIR / "43_yfinance_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
display(status)
display(pd.Series(manifest, name="value").to_frame())
assert status.set_index("ticker").loc["MISSING_DEMO", "status"] == "empty"
assert len(digest) == 64

,ticker,status,rows
0,BANK_A,ok,756
1,EV_A,ok,756
2,TECH_A,ok,756
3,MISSING_DEMO,empty,0


,value
adapter,yfinance-style-offline
yfinance_version,1.5.1
auto_adjust,False
price_fields,"[Close, Adj Close, Volume]"
date_start,2023-01-03
date_end,2025-11-25
sha256,15af57f865410ff035563ea5afea510a41eebb14468767...


## 方法边界与复用结论


- `download` 与 `Ticker.history` 的返回结构和默认值会随版本变化，必须固定参数并记录版本。
- `auto_adjust`、`Close`、`Adj Close` 的语义不能凭字段名猜测；公司行动要与 `actions` 抽查。
- yfinance 是非官方研究接口，限流、空表和部分失败属于数据质量状态。


本课代码验证的是方法和数据契约，不是未来收益。迁移到真实研究前，还需重新核对来源授权、复权、时区、可得时间、交易成本和样本外边界。